# Model Pipeline: Retail Products Classification (Multimodal Dataset)
**Eksperimen**: `06_retail_products_classification`  
**Sumber Data**: `project/dataset/barudownload/retail-products-classification/` (46.229 baris, 21 kategori produk ritel, dilengkapi file gambar foto produk)  
**Tujuan**: Menjalankan EDA kategori, vektorisasi teks & deskripsi barang, semantic retrieval, dan evaluasi performa metrik (Recall@K, MRR).

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Data & Analisis Kategori Ritel

In [2]:
df = pd.read_csv('../data/products.csv')
print(f"Total Sampel Produk: {df.shape[0]} produk")
print(f"Total Kategori Ritel: {df['category'].nunique()}")
df.head()

Total Sampel Produk: 2100 produk
Total Kategori Ritel: 21


,item_id,item_name,category,description,image_path
0,B005NIX12S,Bootstraps Tourquoise Polka (Pair),All Beauty,Are You Tired Of Tucking Your Pants Into Your ...,train/train/B005NIX12S.jpg
1,B001LBXBJY,Laura Mercier Lip Colour Hydrangea ( Creme ) 3...,All Beauty,Gives long wearing & stay-true color Formulat...,train/train/B001LBXBJY.jpg
2,B005WKXAFA,NYC New York Color Smooch Proof Long Wearing L...,All Beauty,LONG WEARING LIP COLOR,train/train/B005WKXAFA.jpg
3,B00643QH8G,BDS - Magic Hair Shorten Tool,All Beauty,Instruction:\nJust like the bun set ones; put ...,train/train/B00643QH8G.jpg
4,B002OQERI4,Travel Set: Liquid Soap + Face Scrub + DDML + ...,All Beauty,Travel Set: 1x Liquid Facial Soap Oily Skin F...,train/train/B002OQERI4.jpg


## 2. Vektorisasi Teks & Mesin Pencarian Produk Ritel

In [3]:
df['document'] = df['item_name'] + ' ' + df['category'] + ' ' + df['description'].fillna('')

vectorizer = TfidfVectorizer(max_features=10000)
tfidf_matrix = vectorizer.fit_transform(df['document'])
print(f"Ukuran Matriks TF-IDF: {tfidf_matrix.shape}")

def search_retail_products(query, top_k=5):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    
    results = []
    for idx in top_idx:
        if sims[idx] > 0:
            results.append({
                'Title': df.iloc[idx]['item_name'],
                'Category': df.iloc[idx]['category'],
                'Similarity': round(float(sims[idx]), 4),
                'Image': df.iloc[idx]['image_path']
            })
    return pd.DataFrame(results)

Ukuran Matriks TF-IDF: (2100, 10000)


## 3. Uji Coba Kueri Pencarian Kebutuhan Pengguna

In [4]:
sample_q = 'electronics cell phone accessory charger cable'
print(f"Mencari: {sample_q}")
search_retail_products(sample_q, top_k=5)

Mencari: electronics cell phone accessory charger cable


,Title,Category,Similarity,Image
0,Samsung Galaxy S Phone AA battery Powered Emer...,Cell Phones & Accessories,0.3962,train/train/9985534263.jpg
1,Premium Accessory Power Pack for your Motorola...,Cell Phones & Accessories,0.3446,train/train/9985634462.jpg
2,Premium Car Charger for your Micro USB The Bes...,Cell Phones & Accessories,0.3164,train/train/9977649308.jpg
3,Samsung Galaxy Precedent Smartphone AA battery...,Cell Phones & Accessories,0.3155,train/train/9967087838.jpg
4,Motorola Adaptable Dual Pocket Desktop Charger...,Cell Phones & Accessories,0.3071,train/train/B0000633KZ.jpg


## 4. Evaluasi Kuantitatif Metrik (Recall@K, Precision@K, MRR)

In [5]:
queries = pd.read_csv('../data/test_queries.csv')
k_list = [1, 3, 5]
recalls = {k: [] for k in k_list}
precisions = {k: [] for k in k_list}
mrrs = []

for _, row in queries.iterrows():
    q = row['query']
    target = row['positive_item']
    
    q_vec = vectorizer.transform([q])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:5]
    retrieved = [df.iloc[i]['item_name'] for i in top_idx]
    
    # MRR
    rank = 0
    for r_idx, item in enumerate(retrieved, 1):
        if item == target:
            rank = r_idx
            break
    mrrs.append(1.0 / rank if rank > 0 else 0.0)
    
    for k in k_list:
        hit = target in retrieved[:k]
        recalls[k].append(1.0 if hit else 0.0)
        precisions[k].append(1.0 / k if hit else 0.0)

print('========== HASIL EVALUASI RETAIL PRODUCTS CLASSIFICATION ==========')
print(f'Recall@1    : {np.mean(recalls[1]):.4f} ({np.mean(recalls[1])*100:.1f}%)')
print(f'Recall@3    : {np.mean(recalls[3]):.4f} ({np.mean(recalls[3])*100:.1f}%)')
print(f'Recall@5    : {np.mean(recalls[5]):.4f} ({np.mean(recalls[5])*100:.1f}%)')
print(f'Precision@3 : {np.mean(precisions[3]):.4f}')
print(f'MRR Score   : {np.mean(mrrs):.4f}')

========== HASIL EVALUASI RETAIL PRODUCTS CLASSIFICATION ==========
Recall@1    : 0.7143 (71.4%)
Recall@3    : 0.9048 (90.5%)
Recall@5    : 0.9048 (90.5%)
Precision@3 : 0.3016
MRR Score   : 0.8095
